# 01 – Inspekcja danych (GWHD 2021)

Cel tego notebooka: **ręcznie obejrzeć jeden obraz i jego adnotacje**, zanim cokolwiek zbudujemy.

Co robimy po kolei:
1. Sprawdzamy środowisko (wersja Pythona, czy jest PyTorch i karta graficzna).
2. Wskazujemy, gdzie leży plik ZIP z danymi, i sprawdzamy, czy istnieje.
3. Zaglądamy do środka ZIP-a **bez rozpakowywania go** (ZIP ma ok. 10 GB).
4. Wczytujemy plik CSV z adnotacjami i oglądamy jego kolumny.
5. Zamieniamy tekst z ramkami na liczby (w tym obsługa etykiety `no_box`).
6. Wczytujemy jeden obraz i rysujemy na nim ramki.

Czego tu **nie** robimy: map gęstości, kafelków, Datasetu PyTorch ani treningu.

> **Pojęcia:**
> *Adnotacja* – ręcznie zaznaczona informacja o obrazie (tu: gdzie są kłosy pszenicy).
> *Ramka (bounding box)* – prostokąt wokół jednego obiektu, zapisany jako 4 liczby: lewy-górny i prawy-dolny róg.

## Krok 1. Środowisko

Wypisujemy wersję Pythona i system. Potem **próbujemy** zaimportować PyTorch – na tym etapie nie jest
potrzebny, więc jego brak to nie błąd, tylko informacja.

*GPU* (karta graficzna) przyspiesza trening sieci; *CUDA* to sposób, w jaki PyTorch z niej korzysta.

In [ ]:
import sys
import platform

print("Python:", sys.version)
print("System:", platform.platform())
print("Interpreter:", sys.executable)  # powinien wskazywać na .venv (lokalnie)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("GPU (CUDA) dostępne:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Karta:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch nie jest zainstalowany - na tym etapie to w porządku.")

## Krok 2. Ścieżka do ZIP-a

Ścieżkę ustalamy tak, żeby działała **i na Windowsie, i w Colabie**:

1. Jeśli ustawiona jest zmienna środowiskowa `GWHD_ZIP` – używamy jej.
2. W przeciwnym razie szukamy pliku `data/raw/gwhd_2021.zip` w katalogu głównym repozytorium.

`pathlib.Path` sam dobiera właściwe ukośniki (`\` na Windowsie, `/` w Colabie).

**Colab:** przed tą komórką wykonaj np.
`import os; os.environ["GWHD_ZIP"] = "/content/drive/MyDrive/gwhd_2021.zip"`

Notebook **niczego nie pobiera** – tylko sprawdza, czy plik jest.

In [ ]:
import os
from pathlib import Path

# Katalog główny repo: notebook zwykle startuje w folderze notebooks/, więc idziemy poziom wyżej.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_ZIP = Path(os.environ.get("GWHD_ZIP", ROOT / "data" / "raw" / "gwhd_2021.zip"))

print("Szukam pliku:", DATA_ZIP)

if not DATA_ZIP.is_file():
    raise FileNotFoundError(
        f"Nie ma pliku: {DATA_ZIP}\n"
        "Umieść ZIP w data/raw/gwhd_2021.zip albo ustaw zmienną GWHD_ZIP (patrz README). "
        "Dalsze komórki wymagają tego pliku."
    )

print(f"OK, plik istnieje ({DATA_ZIP.stat().st_size / 1e9:.2f} GB)")

## Krok 3. Co jest w środku ZIP-a?

Moduł `zipfile` pozwala przeczytać **listę plików** i pojedyncze pliki z archiwum bez rozpakowywania całości.

Szukamy dwóch rodzajów plików:
- **CSV** – tabele z adnotacjami,
- **obrazy** (`.png` / `.jpg`).

In [ ]:
import zipfile

zf = zipfile.ZipFile(DATA_ZIP)  # otwieramy archiwum tylko do odczytu
all_names = zf.namelist()

csv_names = [n for n in all_names if n.lower().endswith(".csv")]
image_names = [n for n in all_names if n.lower().endswith((".png", ".jpg", ".jpeg"))]

print("Plików w ZIP-ie:", len(all_names))
print("\nPliki CSV:")
for n in csv_names:
    print("  ", n)
print(f"\nObrazów: {len(image_names)}. Pierwsze 5:")
for n in image_names[:5]:
    print("  ", n)

## Krok 4. Wczytanie CSV z adnotacjami

Wybieramy plik CSV ze słowem `train` w nazwie (zbiór treningowy); jeśli takiego nie ma – pierwszy z listy.
Możesz go zmienić ręcznie, wpisując inną nazwę do `CSV_NAME`.

`pandas` wczytuje CSV jako *DataFrame* – tabelę z kolumnami, podobną do arkusza w Excelu.

In [ ]:
import pandas as pd

train_csvs = [n for n in csv_names if "train" in n.lower()]
CSV_NAME = train_csvs[0] if train_csvs else csv_names[0]
print("Wczytuję:", CSV_NAME)

with zf.open(CSV_NAME) as f:
    df = pd.read_csv(f)

print("Liczba wierszy:", len(df))
print("Kolumny:", list(df.columns))
df.head()

Jeden wiersz w całości – żeby zobaczyć, jak dokładnie wyglądają zapisane ramki:

In [ ]:
row = df.iloc[0]
for col, value in row.items():
    print(f"{col}: {str(value)[:300]}")  # ucinamy bardzo długie teksty

## Krok 5. Zamiana tekstu z ramkami na liczby

Według dokumentacji GWHD 2021 kolumna z ramkami (zwykle `BoxesString`) ma postać:

```
x_min y_min x_max y_max;x_min y_min x_max y_max;...
```

czyli ramki oddzielone średnikiem, a liczby w ramce – spacją. Współrzędne są w pikselach,
punkt (0, 0) to **lewy górny** róg obrazu.

Obraz bez żadnego kłosa ma zamiast ramek tekst **`no_box`** – wtedy zwracamy pustą listę (0 roślin).

Kolumny wykrywamy automatycznie po nazwie. Jeśli wykrycie zawiedzie, wpisz nazwy ręcznie.

In [ ]:
# Automatyczne wykrycie kolumn (można nadpisać ręcznie, np. BOX_COL = "BoxesString")
IMAGE_COL = next(c for c in df.columns if "image" in c.lower())
BOX_COL = next(c for c in df.columns if "box" in c.lower())
print("Kolumna z nazwą obrazu:", IMAGE_COL)
print("Kolumna z ramkami:", BOX_COL)


def parse_boxes(text):
    """Zamienia tekst 'x1 y1 x2 y2;...' na listę krotek liczb. 'no_box' -> pusta lista."""
    if pd.isna(text) or str(text).strip() == "no_box":
        return []
    boxes = []
    for part in str(text).split(";"):
        if part.strip():
            x1, y1, x2, y2 = (float(v) for v in part.split())
            boxes.append((x1, y1, x2, y2))
    return boxes


df["n_boxes"] = df[BOX_COL].apply(lambda s: len(parse_boxes(s)))

print("\nObrazów z etykietą no_box:", (df[BOX_COL].astype(str).str.strip() == "no_box").sum())
print("Liczba ramek na obraz - podsumowanie:")
print(df["n_boxes"].describe())

## Krok 6. Wczytanie jednego obrazu

Wybieramy obraz z wiersza nr `ROW_INDEX` (zmień liczbę, żeby obejrzeć inny).
Nazwa w CSV to sama nazwa pliku, a w ZIP-ie plik może leżeć w podfolderze – dlatego szukamy
po końcówce ścieżki.

In [ ]:
from io import BytesIO
from PIL import Image

ROW_INDEX = 0  # <- zmień, aby obejrzeć inny obraz

sample = df.iloc[ROW_INDEX]
image_file = sample[IMAGE_COL]
boxes = parse_boxes(sample[BOX_COL])

# Szukamy pliku w ZIP-ie po samej nazwie (bez folderów)
matches = [n for n in image_names if n.split("/")[-1] == image_file]
if not matches:
    raise FileNotFoundError(f"Nie znalazłem obrazu {image_file} w ZIP-ie")
zip_path = matches[0]

img = Image.open(BytesIO(zf.read(zip_path)))
print("Plik w ZIP-ie:", zip_path)
print("Rozmiar (szer. x wys.):", img.size)
print("Tryb kolorów:", img.mode)
print("Liczba ramek (= liczba kłosów):", len(boxes))
if boxes:
    print("Pierwsza ramka (x_min, y_min, x_max, y_max):", boxes[0])
    # Kontrola sensu: czy ramki mieszczą się w obrazie i czy x_max > x_min?
    w, h = img.size
    ok = all(0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h for x1, y1, x2, y2 in boxes)
    print("Wszystkie ramki poprawne i w granicach obrazu:", ok)

## Krok 7. Obraz z ramkami

Rysujemy obraz i nakładamy na niego prostokąty. `matplotlib` potrzebuje lewego górnego rogu
oraz szerokości i wysokości, więc liczymy `x_max - x_min` i `y_max - y_min`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img)
for x1, y1, x2, y2 in boxes:
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=1))
ax.set_title(f"{image_file} - {len(boxes)} ramek")
ax.axis("off")
plt.show()

## Podsumowanie

Jeśli wszystko zadziałało, wiesz już:
- jakie pliki CSV są w zbiorze i jakie mają kolumny,
- jak zapisane są ramki i jak wygląda obraz bez obiektów (`no_box`),
- jaki rozmiar mają obrazy i ile kłosów jest na jednym obrazie.

Liczba ramek na obrazie to właśnie **liczba roślin**, którą później będziemy szacować z mapy gęstości.